# 15.2 Other network models

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Not all network linear programs involve the transportation of things or the minimization
of costs. We describe here three well-known model classes — maximum flow,
shortest path, and transportation/assignment — that use the same kinds of variables and
constraints for different purposes.

**Maximum flow models**

In some network design applications the concern is to send as much flow as possible
through the network, rather than to send flow at lowest cost. This alternative is readily
handled by dropping the balance constraints at the origins and destinations of flow, while
substituting an objective that stands for total flow in some sense.

As a specific example, [Figure 15-5](./15_2_other_network_models.ipynb#fig-15-5) presents a diagram of a simple traffic network.
The nodes and arcs represent intersections and roads; capacities, shown as numbers next
to the roads, are in cars per hour. We want to find the maximum traffic flow that can
enter the network at a and leave at g.

A model for this situation begins with a set of intersections, and symbolic parameters
to indicate the intersections that serve as entrance and exit to the road network:

```ampl
set INTER;
param entr symbolic in INTER;
param exit symbolic in INTER, <> entr;
```

The set of roads is defined as a subset of the pairs of intersections:

```ampl
set ROADS within (INTER diff {exit}) cross (INTER diff {entr});
```

This definition ensures that no road begins at the exit or ends at the entrance.

Next, the capacity and traffic load are defined for each road:

```ampl
set INTER;                      # intersections
param entr symbolic in INTER;             # entrance to road network
param exit symbolic in INTER, <> entr;    # exit from road network
set ROADS within (INTER diff {exit}) cross (INTER diff {entr});
param cap {ROADS} >= 0;                         # capacities
var Traff {(i,j) in ROADS} >= 0, <= cap[i,j];   # traffic loads
maximize Entering_Traff: sum {(entr,j) in ROADS} Traff[entr,j];
subject to Balance {k in INTER diff {entr,exit}}:
sum {(i,k) in ROADS} Traff[i,k] = sum {(k,j) in ROADS} Traff[k,j];
data;
set INTER := a b c d e f g ;
param entr := a ;
param exit := g ;
param:   ROADS:  cap :=
         a b     50,    a     c   100
         b d     40,    b     e    20
         c d     60,    c     f    20
         d e     50,    d     f    60
         e g     70,    f     g    70 ;
```

**[Figure 15-6](./15_2_other_network_models.ipynb#fig-15-6):** Maximum traffic flow model and data (netmax.mod).

```ampl
param cap {ROADS} >= 0;
var Traff {(i,j) in ROADS} >= 0, <= cap[i,j];
```

The constraints say that except for the entrance and exit, flow into each intersection
equals flow out:

```ampl
subject to Balance {k in INTER diff {entr,exit}}:
       sum {(i,k) in ROADS} Traff[i,k]
       = sum {(k,j) in ROADS} Traff[k,j];
```

Given these constraints, the flow out of the entrance must be the total flow through the
network, which is to be maximized:

```ampl
maximize Entering_Traff: sum {(entr,j) in ROADS} Traff[entr,j];
```

We could equally well maximize the total flow into the exit. The entire model, along
with data for the example shown in [Figure 15-5](./15_2_other_network_models.ipynb#fig-15-5), is presented in [Figure 15-6](./15_2_other_network_models.ipynb#fig-15-6). Any linear
programming solver will find a maximum flow of 130 cars per hour.

**Shortest path models**

If you were to use the optimal solution to any of our models thus far, you would have
to send each of the packages, cars, or whatever along some path from a supply (or
entrance) `node` to a demand (or exit) `node`. The values of the decision variables do not
directly say what the optimal paths are, or how much flow must go on each one. Usually
it is not too hard to deduce these paths, however, especially when the network has a regular
or special structure.
If a network has just one unit of supply and one unit of demand, the optimal solution
assumes a quite different nature. The variable associated with each `arc` is either 0 or 1,
and the arcs whose variables have value 1 comprise a minimum-cost path from the supply
`node` to the demand `node`. Often the "costs" are in fact times or distances, so that the
optimum gives a shortest path.

Only a few changes need be made to the maximum flow model of [Figure 15-6](./15_2_other_network_models.ipynb#fig-15-6) to turn
it into a shortest path model. There are still a parameter and a variable associated with
each road from `i` to j, but we call them time[i,j] and Use[i,j], and the sum of
their products yields the objective:

```ampl
param time {ROADS} >= 0;                     # times to travel roads
var Use {(i,j) in ROADS} >= 0;               # 1 iff (i,j) in shortest path
minimize Total_Time: sum {(i,j) in ROADS} time[i,j] * Use[i,j];
```

Since only those variables Use[i,j] on the optimal path equal 1, while the rest are 0,
this sum does correctly represent the total time to traverse the optimal path. The only
other change is the addition of a constraint to ensure that exactly one unit of flow is available
at the entrance to the network:

```ampl
subject to Start:          sum {(entr,j) in ROADS} Use[entr,j] = 1;
```

The complete model is shown in [Figure 15-7](./15_2_other_network_models.ipynb#fig-15-7). If we imagine that the numbers on the arcs
in [Figure 15-5](./15_2_other_network_models.ipynb#fig-15-5) are travel times in minutes rather than capacities, the data are the same;
AMPL finds the solution as follows:

```ampl
ampl: model netshort.mod;
ampl: solve;
MINOS 5.5: optimal solution found.

1 iterations, objective 140
ampl: option omit_zero_rows 1;
ampl: display Use;
Use :=
a b    1
b e    1
e g    1
;
```

The shortest path is a → b → e → g, which takes 140 minutes.

Transportation and assignment models
The best known and most widely used special network structure is the "bipartite"
structure depicted in [Figure 15-8](./15_2_other_network_models.ipynb#fig-15-8). The nodes fall into two groups, one serving as origins
of flow and the other as destinations. Each `arc` connects an origin to a destination.

```ampl
set INTER;                       # intersections
param entr symbolic in INTER;               # entrance to road network
param exit symbolic in INTER, <> entr;      # exit from road network
set ROADS within (INTER diff {exit}) cross (INTER diff {entr});
param time {ROADS} >= 0;                    # times to travel roads
var Use {(i,j) in ROADS} >= 0;              # 1 iff (i,j) in shortest path
minimize Total_Time: sum {(i,j) in ROADS} time[i,j] * Use[i,j];
subject to Start:                                   sum {(entr,j) in ROADS} Use[entr,j] = 1;
subject to Balance {k in INTER diff {entr,exit}}:
sum {(i,k) in ROADS} Use[i,k] = sum {(k,j) in ROADS} Use[k,j];
data;
set INTER := a b c d e f g ;
param entr := a ;
param exit := g ;
param: ROADS:               time :=
              a b    50,    a c     100
              b d    40,    b e      20
              c d    60,    c f      20
              d e    50,    d f      60
              e g    70,    f g      70 ;
```

**[Figure 15-7](./15_2_other_network_models.ipynb#fig-15-7):** Shortest path model and data (netshort.mod).

The minimum-cost transshipment model on this network is known as the transportation
model. The special case in which every origin is connected to every destination was
introduced in [Chapter 3](../03/03.md); an AMPL model and sample data are shown in Figures 3-1a
and 3-1b. A more general example analogous to the models developed earlier in this chapter,
where a set LINKS specifies the arcs of the network, appears in Figures 6-2a and 6-2b.

Every path from an origin to a destination in a bipartite network consists of one `arc`.
Or, to say the same thing another way, the optimal flow along an `arc` of the transportation
model gives the actual amount shipped from some origin to some destination. This property
permits the transportation model to be viewed alternatively as a so-called assignment
model, in which the optimal flow along an `arc` is the amount of something from the origin
that is assigned to the destination. The meaning of assignment in this context can be
broadly construed, and in particular need not involve a shipment in any sense.

One of the more common applications of the assignment model is matching people to
appropriate targets, such as jobs, offices or even other people. Each origin `node` is associated
with one person, and each destination `node` with one of the targets — for example,
with one project. The sets might then be defined as follows:

```ampl
set PEOPLE;
set PROJECTS;
set ABILITIES within (PEOPLE cross PROJECTS);

                     DET


                     FRA
CLEV
                     FRE

GARY                        LAF


                     LAN
PITT
                     STL
                     WIN
```

**[Figure 15-8](./15_2_other_network_models.ipynb#fig-15-8):** Bipartite network.

The set ABILITIES takes the role of LINKS in our earlier models; a pair (i,j) is
placed in this set if and only if person `i` can work on project j.

As one possibility for continuing the model, the supply at `node` `i` could be the number
of hours that person `i` is available to work, and the demand at `node` `j` could be the number
of hours required for project j. Variables Assign[i,j] would represent the number
of hours of person i's time assigned to project j. Also associated with each pair
(i,j) would be a cost per hour, and a maximum number of hours that person `i` could
contribute to job j. The resulting model is shown in [Figure 15-9](./15_2_other_network_models.ipynb#fig-15-9).

Another possibility is to make the assignment in terms of people rather than hours.
The supply at every `node` `i` is 1 (person), and the demand at `node` `j` is the number of people
required for project j. The supply constraints ensure that Assign[i,j] is not
greater than 1; and it will equal 1 in an optimal solution if and only if person `i` is
assigned to project j. The coefficient cost[i,j] could be some kind of cost of assigning
person `i` to project j, in which case the objective would still be to minimize total
cost. Or the coefficient could be the ranking of person `i` for project j, perhaps on a scale
from 1 (highest) to 10 (lowest). Then the model would produce an assignment for which
the total of the rankings is the best possible.

Finally, we can imagine an assignment model in which the demand at each `node` `j` is
also 1; the problem is then to match people to projects. In the objective, cost[i,j]
could be the number of hours that person `i` would need to complete project j, in which
case the model would find the assignment that minimizes the total hours of work needed
to finish all the projects. You can create a model of this kind by replacing all references

```ampl
set PEOPLE;
set PROJECTS;
set ABILITIES within (PEOPLE cross PROJECTS);
param supply {PEOPLE} >= 0;   # hours each person is available
param demand {PROJECTS} >= 0; # hours each project requires
check: sum {i in PEOPLE} supply[i] = sum {j in PROJECTS} demand[j];
param cost {ABILITIES} >= 0;           # cost per hour of work
param limit {ABILITIES} >= 0;          # maximum contributions to projects
var Assign {(i,j) in ABILITIES} >= 0, <= limit[i,j];
minimize Total_Cost:
sum {(i,j) in ABILITIES} cost[i,j] * Assign[i,j];
subject to Supply {i in PEOPLE}:
sum {(i,j) in ABILITIES} Assign[i,j] = supply[i];
subject to Demand {j in PROJECTS}:
sum {(i,j) in ABILITIES} Assign[i,j] = demand[j];
```

**[Figure 15-9](./15_2_other_network_models.ipynb#fig-15-9):** Assignment model (netasgn.mod).

to supply[i] and demand[j] by 1 in [Figure 15-9](./15_2_other_network_models.ipynb#fig-15-9). Objective coefficients representing
rankings are an option for this model as well, giving rise to the kind of assignment
model that we used as an example in Section 3.3.